In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Optional, Tuple
import math
import random
import numpy as np

In [28]:
@dataclass
class GPTConfig:
    vocab_size: int = 100  # Set small for toy data; change as needed
    context: int = 32
    d_model: int = 64
    n_head: int = 4
    n_layer: int = 2
    rope: bool = False
    alibi: bool = False

    @property
    def head_dim(self):
        return self.d_model // self.n_head

In [29]:
def rotary_embed(dim, seq_len, device):
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, device=device) / dim))
    t = torch.arange(seq_len, device=device)
    freqs = torch.einsum('i,j->ij', t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos(), emb.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    x_rot = torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)
    return x_rot.flatten(-2)

In [30]:
class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
        )

    def forward(self, x, mask, cos_sin=None):
        qkv = self.ln1(x)
        if cos_sin is not None:
            cos, sin = cos_sin
            qkv = apply_rope(qkv, cos, sin)
        a, w = self.attn(qkv, qkv, qkv, attn_mask=mask, need_weights=True)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, w

In [31]:
class GPTLate(nn.Module):
    def __init__(self, config: GPTConfig = None):
        super().__init__()
        self.config = config or GPTConfig()
        cfg = self.config
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if not (cfg.rope or cfg.alibi):
            self.pos = nn.Parameter(torch.zeros(1, cfg.context, cfg.d_model))
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embed.weight
        self._mask = torch.triu(torch.ones(cfg.context, cfg.context), 1).bool()

    def forward(self, tokens, return_attn=False):
        B, T = tokens.shape
        if T > self.config.context:
            raise ValueError("Sequence length exceeds model context")
        device = tokens.device
        x = self.embed(tokens)
        if not (self.config.rope or self.config.alibi):
            x = x + self.pos[:, :T, :]
        attns = []
        mask = self._mask[:T, :T].to(device)
        cos_sin = None
        if self.config.rope:
            cos, sin = rotary_embed(self.config.head_dim, T, device)
            cos_sin = (cos, sin)
        for blk in self.blocks:
            x, w = blk(x, mask, cos_sin)
            if return_attn:
                attns.append(w.detach())
        x = self.ln_f(x)
        logits = self.head(x)
        return logits, attns if return_attn else None

    @torch.no_grad()
    def generate(self, prompt_ids: List[int], max_new_tokens: int = 20):
        ids = torch.tensor(prompt_ids, dtype=torch.long)[None]
        for _ in range(max_new_tokens):
            logits, _ = self(ids)
            next_id = logits[0, -1].argmax(-1, keepdim=True)
            ids = torch.cat((ids, next_id[None]), dim=1)
            if ids.shape[1] >= self.config.context:
                ids = ids[:, -self.config.context :]
        return ids.squeeze().tolist()

In [32]:
# For a real project, use a proper tokenizer. Here, we use a toy one for demonstration.
class ToyTokenizer:
    def __init__(self):
        self.vocab = {chr(i+65): i for i in range(26)}  # A-Z
        self.inv_vocab = {i: chr(i+65) for i in range(26)}
    def encode(self, s):
        return [self.vocab.get(c.upper(), 0) for c in s if c.isalpha()]
    def decode(self, ids):
        return ''.join(self.inv_vocab.get(i, '?') for i in ids)
tok = ToyTokenizer()

In [33]:
# Let's create a simple dataset: sequences of random letters
def make_dataset(num_samples=1000, seq_len=16):
    data = []
    for _ in range(num_samples):
        s = ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ') for _ in range(seq_len))
        ids = tok.encode(s)
        data.append(ids)
    return data

dataset = make_dataset(500, 16)
print("Sample:", dataset[0])

Sample: [14, 25, 3, 25, 8, 17, 18, 4, 0, 2, 18, 17, 15, 24, 15, 4]


In [34]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, data, context):
        self.samples = []
        for seq in data:
            for i in range(len(seq) - context):
                chunk = seq[i:i+context+1]
                self.samples.append(chunk)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        chunk = self.samples[idx]
        return torch.tensor(chunk[:-1]), torch.tensor(chunk[1:])

context = 8
train_ds = CharDataset(dataset, context)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)

In [35]:
cfg = GPTConfig(
    vocab_size=26,
    context=context,
    d_model=64,
    n_head=4,
    n_layer=2,
    rope=False,
    alibi=False
)
model = GPTLate(cfg)
print(model)

GPTLate(
  (embed): Embedding(26, 64)
  (blocks): ModuleList(
    (0-1): 2 x Block(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
    )
  )
  (ln_f): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=26, bias=False)
)


In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits, _ = model(xb)
        loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), yb.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_dl):.4f}")

Epoch 1, Loss: 6.0995
Epoch 2, Loss: 3.2736
Epoch 3, Loss: 3.2547


In [37]:
prompt = "HELLO"
input_ids = tok.encode(prompt)
output_ids = model.generate(input_ids, max_new_tokens=20)
print("Prompt:", prompt)
print("Generated:", tok.decode(output_ids))

Prompt: HELLO
Generated: TTCVVRRL


In [38]:
import matplotlib.pyplot as plt
with torch.no_grad():
    inp = torch.tensor([input_ids], dtype=torch.long).to(device)
    logits, attns = model(inp, return_attn=True)
    attn = attns[-1][0].mean(0).cpu().numpy()  # [T, T]
    plt.imshow(attn, cmap='viridis')
    plt.title("Attention (last layer, mean heads)")
    plt.xlabel("Input position")
    plt.ylabel("Output position")
    plt.colorbar()
    plt.show()

RuntimeError: Numpy is not available